# Explore the movie data

This notebook is for exploratory analysis after the normalized movie tables have passed the data-quality profiling step. It loads the local cache without making API requests.


In [ ]:
from IPython.display import display
import pandas as pd

from streamvault.cache import load_cached_records
from streamvault.config import CONFIG
from streamvault.normalize import normalize_movie_records

records = list(load_cached_records(CONFIG.cache_file).values())
tables = normalize_movie_records(records)
movies = tables.movies

frames = {
    name: getattr(tables, name)
    for name in (
        "movies", "genres", "movie_genres", "companies",
        "movie_companies", "countries", "movie_origin_countries",
        "movie_production_countries", "languages",
        "movie_spoken_languages", "collections", "people",
        "cast_credits", "crew_credits",
    )
}

print(f"Loaded {len(records):,} cached movie records")
if not records:
    print("No cached movies yet. Run `uv run streamvault` first.")


## 3. Numeric movie fields and unusual values

Use quantiles and interquartile-range (IQR) bounds to locate values worth checking. These bounds are exploratory, not universal validity rules; budget and revenue may be zero or absent, for example, and an extreme value may be legitimate.


In [ ]:
identifier_columns = {"movie_id", "collection_id"}
numeric_movie_columns = [
    column
    for column in movies.select_dtypes(include="number").columns
    if column not in identifier_columns
]

if numeric_movie_columns:
    numeric_summary = movies[numeric_movie_columns].describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.90, 0.99]
    ).T
    display(numeric_summary)
else:
    print("No numeric movie measures are available.")

outlier_rows = []
for column in numeric_movie_columns:
    values = pd.to_numeric(movies[column], errors="coerce")
    valid_values = values.dropna()
    if valid_values.empty:
        continue

    q1 = valid_values.quantile(0.25)
    q3 = valid_values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    unusual = values.lt(lower_bound) | values.gt(upper_bound)
    examples = movies.loc[unusual, ["movie_id", column]].head(5)

    outlier_rows.append(
        {
            "field": column,
            "iqr_lower_bound": lower_bound,
            "iqr_upper_bound": upper_bound,
            "values_outside_iqr": int(unusual.sum()),
            "negative_values": int(valid_values.lt(0).sum()),
            "example_movie_values": [
                f"{row.movie_id}: {row_value}"
                for row, row_value in zip(
                    examples.itertuples(index=False),
                    examples[column],
                    strict=True,
                )
            ],
        }
    )

outlier_review = pd.DataFrame(outlier_rows)
display(outlier_review)


## 4. Release dates and common categories

Review coverage and the observed range before defining date rules. Frequency tables help identify dominant categories, spelling/label variants, or unexpected missing values.


In [ ]:
if "release_date" in movies.columns:
    release_dates = movies["release_date"].dropna()
    print(f"Movies with a release date: {len(release_dates):,} / {len(movies):,}")
    if not release_dates.empty:
        print(f"Observed release-date range: {release_dates.min()} to {release_dates.max()}")
        release_year_counts = (
            release_dates.dt.year.value_counts().sort_index().rename("movie_count")
        )
        print("Earliest release years:")
        display(release_year_counts.head(15).to_frame())
        print("Latest release years:")
        display(release_year_counts.tail(15).to_frame())
else:
    print("No normalized release_date column is available.")

category_columns = [
    column
    for column in ("status", "original_language", "adult", "video")
    if column in movies.columns
]
category_rows = []
for column in category_columns:
    counts = movies[column].value_counts(dropna=False).head(15)
    for value, count in counts.items():
        category_rows.append(
            {
                "field": column,
                "value": value,
                "movie_count": int(count),
                "percent_of_movies": (
                    round(count / len(movies) * 100, 2) if len(movies) else pd.NA
                ),
            }
        )

category_profile = pd.DataFrame(category_rows)
display(category_profile)


## 5. Profile relationships

Compare relationship counts to all movies, including movies with zero links. Then inspect the most common genres. The left join preserves links even if a reference label is missing; the match count makes that case visible.


In [ ]:
relationship_tables = {
    "genres": tables.movie_genres,
    "production companies": tables.movie_companies,
    "origin countries": tables.movie_origin_countries,
    "production countries": tables.movie_production_countries,
    "spoken languages": tables.movie_spoken_languages,
    "cast credits": tables.cast_credits,
    "crew credits": tables.crew_credits,
}

relationship_rows = []
for relationship_name, links in relationship_tables.items():
    counts_per_movie = links.groupby("movie_id").size().reindex(movies["movie_id"], fill_value=0)
    relationship_rows.append(
        {
            "relationship": relationship_name,
            "link_rows": len(links),
            "movies_with_at_least_one": int(counts_per_movie.gt(0).sum()),
            "movies_with_zero": int(counts_per_movie.eq(0).sum()),
            "median_links_per_movie_including_zero": (
                float(counts_per_movie.median()) if len(movies) else pd.NA
            ),
            "maximum_links_for_one_movie": (
                int(counts_per_movie.max()) if len(movies) else pd.NA
            ),
        }
    )

relationship_profile = pd.DataFrame(relationship_rows)
display(relationship_profile)

genre_counts = (
    tables.movie_genres.groupby("genre_id")
    .size()
    .rename("movie_count")
    .reset_index()
    .merge(
        tables.genres,
        on="genre_id",
        how="left",
        validate="many_to_one",
        indicator="genre_match",
    )
    .sort_values("movie_count", ascending=False)
)
display(genre_counts.head(20))
print("Genre-reference matches:")
display(genre_counts["genre_match"].value_counts(dropna=False).to_frame("link_rows"))


## 6. Turn the profile into follow-up work

Use this checklist to complete the profiling cycle:

- **Confirm grain:** check that every table has the intended one-row-per-entity, one-row-per-link, or one-row-per-credit meaning.
- **Investigate missingness:** determine whether each missing field is optional, not collected, or a pipeline defect. Do not fill missing values without a reason.
- **Review unusual measures:** inspect IQR examples and negative counts against source records and domain context; do not treat statistical outliers as automatic errors.
- **Review date/category coverage:** verify unexpected periods, labels, and missing categories with a data owner before setting accepted ranges.
- **Check relationships:** resolve unmatched reference IDs and understand movies with no links where coverage is expected.
- **Write explicit rules:** once business expectations are agreed, add named checks and thresholds to the project's staged data checks rather than silently cleaning this notebook's working copy.

Re-run from the top whenever the cache changes. The tables and summaries are generated from the local cache each time; no intermediate output is written back to the data files.
